# Stage 1 - Preprocessing & Validation Setup

This notebook contains:
1. **Environment & Paths**: Verify working directories and imports
2. **Dynamic Random Sampling**: Inspect random rows from the multi-million record TSV files instantly
3. **Text & Address Normalization**: Test name suffix stripping, address expansion, and French accents
4. **Local Validation Split**: Create a fast, balanced 10,000-query validation slice with ground truth
5. **Validation Verification**: Verify singleton ratio and entity matching structures
6. **Full Dataset Preprocessing**: Batch-process all train and test files into `dataset/preprocessed/`

In [ ]:
# 1. Environment & Path Setup
import os
import sys
import random
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print(f"Python Version : {sys.version.split()[0]}")
print(f"Pandas Version : {pd.__version__}")
print(f"Root Directory : {ROOT_DIR}")

In [ ]:
# 2. Fast Dynamic Random Sampling from Dataset
def get_random_sample(filepath, n=5):
    """Pick n random rows from a large TSV file instantly using random byte seeks."""
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        header = f.readline().strip().split("\t")
        file_size = os.path.getsize(filepath)
        rows = []
        for _ in range(n * 5):
            pos = random.randint(0, max(0, file_size - 2000))
            f.seek(pos)
            f.readline()  # discard partial line
            line = f.readline().strip()
            if line:
                parts = line.split("\t")
                if len(parts) == len(header):
                    rows.append(parts)
            if len(rows) >= n:
                break
    return pd.DataFrame(rows[:n], columns=header)

DATA_DIR = "../student_resource/dataset/train"

print("Random Source 1 sample (changes every run):")
s1_sample = get_random_sample(f"{DATA_DIR}/train_source1.tsv", n=5)
display(s1_sample)

print("\nRandom Ground Truth sample (changes every run):")
gt_sample = get_random_sample(f"{DATA_DIR}/train_ground_truth.tsv", n=5)
display(gt_sample)

In [ ]:
# 3. Test Preprocessing Pipeline on Random Real Records + France Test Case
from src.preprocessing import preprocess_record

# Real records sampled dynamically from Cell 2
real_records = s1_sample.to_dict("records")

# Include a French test case (from test set distribution)
france_sample = {
    "entity_id": "TEST-FR-01",
    "business_name": "Société Générale S.A.R.L.",
    "business_address": "12 Bd Haussmann, 75009 Paris",
    "country": "France",
}
all_samples = real_records + [france_sample]

print(f"=== Preprocessing {len(all_samples)} Records (Dynamic Real Samples + French Test Case) ===\n")
for s in all_samples:
    cleaned = preprocess_record(s)
    print(f"ID: {s['entity_id']} ({s.get('country', '')})")
    print(f"  Original : {s.get('business_name', '')} | {s.get('business_address', '')}")
    print(f"  Cleaned  : {cleaned['clean_name']} [Core: {cleaned['core_name']}] (Suffix: '{cleaned['legal_suffix']}')")
    print(f"  Address  : {cleaned['clean_address']}")
    print(f"  Postal   : {cleaned['postal_code']}")
    print("-" * 80)

In [ ]:
# 4. Generate Fast Stratified Local Validation Split (10,000 S1 queries)
!python ../scripts/create_val_split.py --data-dir ../student_resource/dataset/train --out-dir ../dataset/val_split --num-s1 10000 --num-distractors 30000

In [ ]:
# 5. Verify the Generated Validation Dataset
val_s1 = pd.read_csv("../dataset/val_split/val_source1.tsv", sep="\t")
val_gt = pd.read_csv("../dataset/val_split/val_ground_truth.tsv", sep="\t")

singletons = (val_gt["matched_entity_ids"].isna() | (val_gt["matched_entity_ids"] == "")).sum()
print(f"Validation Source 1 rows    : {len(val_s1):,}")
print(f"Validation Ground Truth rows : {len(val_gt):,}")
print(f"Singletons (no match)        : {singletons:,} ({singletons / len(val_gt) * 100:.1f}%)")
display(val_s1.head(3))
display(val_gt.head(3))

### 6. Preprocess Complete Dataset (Train & Test)

Runs streaming chunked normalization across all 6 files:
- `train_source1.tsv`, `train_source2.tsv`, `train_source3.tsv`
- `test_source1.tsv`, `test_source2.tsv`, `test_source3.tsv`

Outputs are saved in `dataset/preprocessed/` with clean names, core names, legal suffixes, standardized addresses, and postal codes.

In [ ]:
# Run high-throughput streaming preprocessing across the complete dataset
!python ../scripts/preprocess_dataset.py --input-dir ../student_resource/dataset --output-dir ../dataset/preprocessed --split all --chunksize 150000